<!-- Copyright (c) 2026 João Frazão. SPDX-License-Identifier: Apache-2.0 -->
# 3. MDO multiobjetivo e frente de Pareto

`api.mdo_study` com NSGA-II no problema ZDT1 (Zitzler, Deb & Thiele, *Evol. Comput.* 8(2), 2000), cuja frente verdadeira é f₂ = 1 − √f₁. Mostra a frente obtida, a verdadeira e o **joelho** escolhido (Das 1999). O mesmo bloco `mdo:` serve para uma missão real com `api.mdo(missão, …)` (`docs/MDO.md`).

Definições baratas (população 40, 60 gerações, 5 variáveis): ~15 s; a função é analítica.

In [ ]:
import os, tempfile
from pathlib import Path
import matplotlib
matplotlib.use("Agg")  # sem ecrã (CI); no Jupyter as figuras aparecem na mesma com display
import matplotlib.pyplot as plt
from acdt import api

# raiz do repositório (o notebook corre em examples/notebooks ou na raiz)
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").is_file())
print(api.version_info())
import math
import numpy as np

In [ ]:
N = 5

def zdt1(p):
    xs = [float(p[f"x{i}"]) for i in range(1, N + 1)]
    g = 1.0 + 9.0 * sum(xs[1:]) / (N - 1)
    return {"f1": xs[0], "f2": g * (1.0 - math.sqrt(xs[0] / g))}

model = api.function_model(zdt1, name="ZDT1")
spec = {"design_variables": [{"path": f"x{i}", "lower": 0, "upper": 1, "x0": 0.5, "source": "ZDT1"}
                             for i in range(1, N + 1)],
        "objectives": ["f1", "f2"], "strategy": "nsga2",
        "options": {"pop_size": 40, "max_gen": 60, "seed": 3}}
out = Path(tempfile.mkdtemp(prefix="acdt_nb3_"))
r = api.mdo_study(spec, model, out)
ev = {e["id"]: e for e in r["evaluations"]}
F = np.array(sorted((ev[i]["objectives"]["f1"], ev[i]["objectives"]["f2"]) for i in r["pareto"]))
print(len(r["evaluations"]), "avaliações;", len(F), "projetos na frente")

In [ ]:
gap = F[:, 1] - (1 - np.sqrt(F[:, 0]))
print(f"distância mediana à frente verdadeira: {np.median(gap):.4f}")
knee = ev[r["knee"]["id"]]["objectives"] if r.get("knee") else None
print("joelho:", knee)

In [ ]:
f1 = np.linspace(0, 1, 200)
fig, ax = plt.subplots(figsize=(5.5, 4))
ax.plot(f1, 1 - np.sqrt(f1), "k-", lw=1, label="frente verdadeira")
ax.plot(F[:, 0], F[:, 1], "o", ms=4, label="NSGA-II (acdt)")
if knee:
    ax.plot(knee["f1"], knee["f2"], "r*", ms=14, label="joelho")
ax.set_xlabel("f1"); ax.set_ylabel("f2"); ax.legend(); ax.set_title("ZDT1")
plt.show()
print("ficheiros:", sorted(p.name for p in out.iterdir()))